In [1]:
import re
from collections import Counter

# Input containing two documents
text = """
AI in Healthcare
Artificial intelligence helps doctors analyze medical images.
Machine learning can support disease prediction.

Cyber Security
Cyber security protects computer systems.
Encryption helps protect sensitive information.
"""

# Split input into non-empty lines
lines = [
    line.strip()
    for line in text.strip().splitlines()
    if line.strip()
]

# Simple probabilistic scoring model
def classify_line(line):
    words = line.split()
    word_count = len(words)

    # Estimate heading probability
    if word_count <= 5 and not line.endswith((".", "!", "?")):
        heading_score = 0.8
    else:
        heading_score = 0.2

    # Estimate body-text probability
    if word_count > 5 and line.endswith("."):
        body_score = 0.8
    else:
        body_score = 0.2

    # Estimate separator probability
    if re.search(r"^\s*[-=*_]{3,}\s*$", line):
        separator_score = 0.95
    else:
        separator_score = 0.05

    scores = {
        "HEADING": heading_score,
        "BODY": body_score,
        "SEPARATOR": separator_score
    }

    return max(scores, key=scores.get), scores

# FSM states:
# OUTSIDE = waiting for a document
# INSIDE = collecting document content

state = "OUTSIDE"
documents = []
current_document = []

print("Line Classification:\n")

for line in lines:
    label, scores = classify_line(line)

    print(f"Line: {line}")
    print(f"Predicted Label: {label}")
    print("Scores:", scores)
    print()

    # Finite State Machine transitions
    if label == "SEPARATOR":
        if current_document:
            documents.append(current_document)
            current_document = []
        state = "OUTSIDE"

    elif label == "HEADING":
        # A new heading begins a new document
        if current_document:
            documents.append(current_document)

        current_document = [line]
        state = "INSIDE"

    else:
        # Body text belongs to the current document
        if state == "OUTSIDE":
            current_document = []

        current_document.append(line)
        state = "INSIDE"

# Save the final document
if current_document:
    documents.append(current_document)

print("=" * 50)
print("Detected Documents:")
print("=" * 50)

for i, document in enumerate(documents, start=1):
    print(f"\nDocument {i}:")
    print("\n".join(document))

Line Classification:

Line: AI in Healthcare
Predicted Label: HEADING
Scores: {'HEADING': 0.8, 'BODY': 0.2, 'SEPARATOR': 0.05}

Line: Artificial intelligence helps doctors analyze medical images.
Predicted Label: BODY
Scores: {'HEADING': 0.2, 'BODY': 0.8, 'SEPARATOR': 0.05}

Line: Machine learning can support disease prediction.
Predicted Label: BODY
Scores: {'HEADING': 0.2, 'BODY': 0.8, 'SEPARATOR': 0.05}

Line: Cyber Security
Predicted Label: HEADING
Scores: {'HEADING': 0.8, 'BODY': 0.2, 'SEPARATOR': 0.05}

Line: Cyber security protects computer systems.
Predicted Label: HEADING
Scores: {'HEADING': 0.2, 'BODY': 0.2, 'SEPARATOR': 0.05}

Line: Encryption helps protect sensitive information.
Predicted Label: HEADING
Scores: {'HEADING': 0.2, 'BODY': 0.2, 'SEPARATOR': 0.05}

Detected Documents:

Document 1:
AI in Healthcare
Artificial intelligence helps doctors analyze medical images.
Machine learning can support disease prediction.

Document 2:
Cyber Security

Document 3:
Cyber security 